In [1]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv',
    'car_fuel_efficiency_2026.csv')

import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

df = pd.read_csv('car_fuel_efficiency_2026.csv').fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

feats = [c for c in df.columns if c != 'fuel_efficiency_mpg']
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train[feats].to_dict(orient='records'))
X_val = dv.transform(df_val[feats].to_dict(orient='records'))

In [2]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [3]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print(root_mean_squared_error(y_val, rf.predict(X_val)))

1.8332197358745623


In [4]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    print(n, round(root_mean_squared_error(y_val, rf.predict(X_val)), 3))

10 1.833
50 1.774
100 1.77
150 1.769


In [8]:
print(df.shape, list(df.columns))
print(X_train.shape)

(10000, 11) ['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors', 'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight', 'acceleration', 'fuel_efficiency_mpg']
(6000, 16)


In [5]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    print(d, np.mean(scores))

10 1.7543924943801008
15 1.7842243445385957
20 1.7867711878395158
25 1.7863054936840876


In [6]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out()).sort_values(ascending=False)
print(imp.head(10))
print(imp[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']])

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
dtype: float64
vehicle_weight         0.203313
horsepower             0.078271
acceleration           0.054998
engine_displacement    0.061281
dtype: float64


In [7]:
fn = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=fn)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=fn)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    params = {'eta': eta, 'max_depth': 6, 'min_child_weight': 1,
              'objective': 'reg:squarederror', 'nthread': 8,
              'seed': 1, 'verbosity': 1}
    res = {}
    xgb.train(params, dtrain, num_boost_round=100, evals=watchlist,
              evals_result=res, verbose_eval=False)
    print(eta, res['val']['rmse'][-1], min(res['val']['rmse']))

0.3 1.8264579949460302 1.7320478064805087
0.1 1.7248100768778967 1.7121714763059768
